# 05 — Final Model Comparison

> Part of a 5-notebook pipeline (run in order, each a separate kernel session — this is deliberate, see the note in the main README/first notebook):
> 1. `01_preprocessing_and_split.ipynb` — load, clean, dedupe, handle class imbalance, train/val/test split
> 2. `02_classical_ml.ipynb` — TF-IDF + Random Forest / Logistic Regression / Naive Bayes
> 3. `03_rnn_family.ipynb` — Word2Vec + SimpleRNN / GRU / LSTM (+ bidirectional variants)
> 4. `04_bert.ipynb` — BERT-base fine-tuning
> 5. `05_final_comparison.ipynb` — combined results across all 10 models
>
> **Why separate notebooks/kernels:** TensorFlow never releases GPU memory once it claims it, and PyTorch (used by BERT) initializing its own CUDA context in the same process can cause a hard, traceback-less kernel crash rather than a clean error. Keeping RNN training (TensorFlow) and BERT training (PyTorch) in different kernel processes avoids that entirely, and also caps how much of a crash costs you — a crash in notebook 4 no longer takes down anything from notebooks 1–3.
> All shared state (cleaned data, splits, trained models, results) is handed off between notebooks via files on disk, not kernel memory.

This notebook only reads `tuning_log.csv` / `final_results.csv` — run whichever of notebooks 2/3/4 you want included first.

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

TUNING_LOG_PATH = 'tuning_log.csv'
FINAL_RESULTS_PATH = 'final_results.csv'

## 15. Final Model Comparison

All 10 required models (3 classical + 6 RNN-family + BERT), each evaluated at its best hyperparameter configuration, side by side on the held-out test set.

In [ ]:
# Reload from disk explicitly -- this way Section 15 works correctly even if Sections 12/13/14
# were run across several different kernel sessions, as long as each of those sessions ran
# Section 1.1 (which persists tuning_log.csv / final_results.csv after every log_run/record_final_result call).
tuning_log_df = pd.read_csv(TUNING_LOG_PATH)
final_results_df = pd.read_csv(FINAL_RESULTS_PATH).sort_values('test_f1_macro', ascending=False).reset_index(drop=True)
final_results_df

In [ ]:
plt.figure(figsize=(10,6))
plot_df = final_results_df.melt(id_vars='model', value_vars=['test_accuracy', 'test_f1_macro', 'test_f1_weighted'],
                                 var_name='metric', value_name='score')
sns.barplot(data=plot_df, x='score', y='model', hue='metric')
plt.title('Model comparison on the test set')
plt.xlabel('score')
plt.ylabel('')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
best_model_row = final_results_df.iloc[0]
worst_model_row = final_results_df.iloc[-1]

print(f"Best-performing model:  {best_model_row['model']}  (test macro-F1 = {best_model_row['test_f1_macro']:.4f}, accuracy = {best_model_row['test_accuracy']:.4f})")
print(f"Worst-performing model: {worst_model_row['model']}  (test macro-F1 = {worst_model_row['test_f1_macro']:.4f}, accuracy = {worst_model_row['test_accuracy']:.4f})")

In [ ]:
# Full hyperparameter tuning log (every config x every model, with validation metrics) -- for the report appendix.
tuning_log_df.sort_values(['model', 'val_f1_macro'], ascending=[True, False]).reset_index(drop=True)

### Notes for the report

- **Accuracy vs. macro-F1:** given the residual class imbalance (even after capping), always lead with macro-F1 when calling a model "best" — a model can post a high accuracy just by nailing the majority classes while quietly failing on `Neural and Evolutionary Computing`. Check the per-class confusion matrices above for exactly this pattern.
- **BERT vs. everything else:** BERT-base is expected to win on raw performance (contextual embeddings capture more nuance than averaged Word2Vec or bag-of-n-grams TF-IDF), but at dramatically higher training cost. This cost/benefit trade-off is worth a paragraph in the discussion section — is the accuracy gain worth 10-50x the training time, for this specific use case?
- **Limitations to state explicitly:** (1) only 6 of 15 arXiv categories used, (2) majority classes were capped/subsampled rather than trained on their full size, (3) GloVe was not used as a third representation, (4) digits were stripped during cleaning, losing some CV-relevant numeric tokens.
